## Indicators on a chart

Every function of `probatum.ta` returns one value a bar, so what it computes goes on a chart
as it is. Here they are by family on daily Bitcoin bars of 2024 and 2025: averages and
channels over the candles, oscillators on panels of their own with their levels, the
indicators that read volume, and those that name a regime. Each chart holds both years and
opens on the last 180 days.

In [1]:
import pathlib

import exchange_router
import numpy
import probatum

print(f"probatum {probatum.__version__}, exchange-router {exchange_router.__version__}")

probatum 3.0.0, exchange-router 3.3.0


### The bars

Binance's hourly BTCUSDT, fetched once into `bars/`, and resampled to days: each day's
first open, highest high, lowest low, last close and summed volume.

In [2]:
PATH = pathlib.Path("../bars/binance-spot-btcusdt-1h-2024-2025")

if PATH.exists():
    hourly = probatum.Data.load(path=PATH)
else:
    with exchange_router.Router.local(exchanges=["binance"]) as router:
        hourly = probatum.Data.from_router(router, exchange="binance", market="spot",
                                           interval="1h", symbols=["BTCUSDT"],
                                           start="2024-01-01", end="2025-12-31")
    hourly.save(path=PATH)

days = hourly.resample(interval="1d")

high, low, close, volume = days.high, days.low, days.close, days.volume

days

Data(BTCUSDT, 1d, 731 bars from 2024-01-01 00:00 to 2025-12-31 00:00)

### Averages and channels

An exponential and a simple average over the candles, and Bollinger's channel shaded
between its edges. A mark with no `panel=` goes over the candles where it leaves them at
least half the price panel.

In [3]:
bands = probatum.ta.bbands(close, length=20, deviations=2.0)

days.chart(
    probatum.chart.marks.Band(bands.upper, lower=bands.lower, name="Bollinger 20, 2",
                              fill="#8b97a51f"),
    probatum.chart.marks.Line(probatum.ta.ema(close, length=21), name="EMA 21"),
    probatum.chart.marks.Line(probatum.ta.sma(close, length=100), name="SMA 100",
                              style="dashed"),
    focus=180,
    title="Averages and a channel, BTCUSDT 1d, binance spot",
)

Chart(bars=731, panels=2)

### Oscillators

Each on a panel of its own, pinned to its range where it has one, with the levels that make
it readable; `panels=` sets their weights and hides the volume to give them the room. The
RSI is shaded only where it is past 70 or 30, deeper as it goes further, and the MACD's
histogram takes the colour of its sign.

In [4]:
rsi   = probatum.ta.rsi(close, length=14)
macd  = probatum.ta.macd(close, fast=12, slow=26, signal=9)
stoch = probatum.ta.stoch(high, low=low, close=close, length=14, smooth=3)

days.chart(
    probatum.chart.marks.Line(rsi, name="RSI 14", panel="RSI"),
    probatum.chart.marks.Level(value=70.0, panel="RSI", style="dotted"),
    probatum.chart.marks.Level(value=30.0, panel="RSI", style="dotted"),
    probatum.chart.marks.Band(rsi, lower=70.0, only="above", panel="RSI",
                              fill=("#f2364500", "#f2364566")),
    probatum.chart.marks.Band(rsi, lower=30.0, only="below", panel="RSI",
                              fill=("#08998100", "#08998166")),
    probatum.chart.marks.Histogram(macd.histogram, name="MACD 12/26/9", panel="MACD",
                                   color=numpy.where(macd.histogram >= 0.0, "#089981",
                                                     "#f23645")),
    probatum.chart.marks.Line(macd.line, name="MACD", panel="MACD", width=1),
    probatum.chart.marks.Line(macd.signal, name="signal", panel="MACD", width=1,
                              style="dashed"),
    probatum.chart.marks.Line(stoch.k, name="%K 14", panel="stochastic", width=1),
    probatum.chart.marks.Line(stoch.d, name="%D 3", panel="stochastic", width=1,
                              style="dashed"),
    probatum.chart.marks.Level(value=80.0, panel="stochastic", style="dotted"),
    probatum.chart.marks.Level(value=20.0, panel="stochastic", style="dotted"),
    panels=[probatum.chart.marks.Panel(name="volume", show=False),
            probatum.chart.marks.Panel(name="RSI", weight=2.0, range=(0.0, 100.0)),
            probatum.chart.marks.Panel(name="MACD", weight=2.0),
            probatum.chart.marks.Panel(name="stochastic", weight=2.0, range=(0.0, 100.0))],
    focus=180,
    title="Oscillators, BTCUSDT 1d, binance spot",
    height=820,
)

Chart(bars=731, panels=4)

### What volume says

A volume-weighted average over the candles, the running balance of volume on rising and
falling days, and the money flow index, an RSI of price weighted by volume.

In [5]:
days.chart(
    probatum.chart.marks.Line(probatum.ta.vwma(close, volume=volume, length=20),
                              name="VWMA 20"),
    probatum.chart.marks.Line(probatum.ta.obv(close, volume=volume) / 1e3,
                              name="OBV, thousands", panel="OBV"),
    probatum.chart.marks.Line(probatum.ta.mfi(high, low=low, close=close, volume=volume,
                                              length=14), name="MFI 14", panel="MFI"),
    probatum.chart.marks.Level(value=80.0, panel="MFI", style="dotted"),
    probatum.chart.marks.Level(value=20.0, panel="MFI", style="dotted"),
    panels=[probatum.chart.marks.Panel(name="MFI", range=(0.0, 100.0))],
    focus=180,
    title="Volume, BTCUSDT 1d, binance spot",
)

Chart(bars=731, panels=4)

### Regime and structure

The supertrend's edge in force, coloured by the side it holds, Donchian's channel of the
last 20 days' extremes, the ADX with its two directional lines, and the close as a z-score
of its last 50 days.

In [6]:
trend = probatum.ta.supertrend(high, low=low, close=close, length=10, multiplier=3.0)
donchian = probatum.ta.donchian(high, low=low, length=20)
adx = probatum.ta.adx(high, low=low, close=close, length=14)

days.chart(
    probatum.chart.marks.Band(donchian.upper, lower=donchian.lower, name="Donchian 20",
                              fill="#8b97a514"),
    probatum.chart.marks.Line(trend.middle, name="supertrend 10, 3", width=2,
                              color=numpy.where(trend.direction > 0.0, "#089981",
                                                "#f23645")),
    probatum.chart.marks.Line(adx.adx, name="ADX 14", panel="ADX"),
    probatum.chart.marks.Line(adx.plus, name="+DI", panel="ADX", width=1, style="dashed"),
    probatum.chart.marks.Line(adx.minus, name="-DI", panel="ADX", width=1, style="dotted"),
    probatum.chart.marks.Level(value=25.0, panel="ADX", style="dotted"),
    probatum.chart.marks.Line(probatum.ta.zscore(close, length=50), name="z 50",
                              panel="z-score"),
    probatum.chart.marks.Level(value=2.0, panel="z-score", style="dotted"),
    probatum.chart.marks.Level(value=-2.0, panel="z-score", style="dotted"),
    focus=180,
    title="Regime and structure, BTCUSDT 1d, binance spot",
    height=680,
)

Chart(bars=731, panels=4)